# Merged 14-Class ECG Classifier (PTB-XL + Chapman)

Trains ECGNetJoint on **60,914 records** (18,524 PTB-XL + 42,390 Chapman-Shaoxing).
Adds **AFIB** (48 → 9,840 samples) and **STACH** (4 → 7,255 samples) to the 12-class model.

---
## Runtime guide

| Cell | Runtime | Duration | What it does |
|------|---------|----------|--------------|
| 1 | **CPU** | 1 min | Mount Drive + install deps |
| 2 | **CPU** | 1 min | Copy scripts from Drive |
| 3 | **CPU** | ~1.5 hrs | Download Chapman from PhysioNet |
| 4 | **CPU** | ~10 min | Build Chapman index + save tar to Drive |
| 5 | **GPU** | 1 min | Switch to GPU — check + mount Drive |
| 6 | **GPU** | ~20 min | Restore Chapman + copy PTB-XL to SSD |
| 7 | **GPU** | ~1-2 hrs | Run merged training |
| 8 | **GPU** | 1 min | Save model to Drive |

**Before running:** Upload these 4 scripts to `MyDrive/EKG_models/`:
`multilabel_merged.py`, `multilabel_classifier.py`, `cnn_classifier.py`, `dataset_chapman.py`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1  ──  CPU runtime  |  Mount Drive + install dependencies
# ─────────────────────────────────────────────────────────────────────────────
import time
t0 = time.time()
print('=== Cell 1 started: Mount Drive + install deps ===')

from google.colab import drive
print('Mounting Drive...')
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

import subprocess, sys
print('Installing wfdb, scipy, scikit-learn...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)
print('Packages installed.')

import torch
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}  (expected False on CPU runtime)')
print(f'=== Cell 1 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2  ──  CPU runtime  |  Copy scripts from Drive to /content/
# ─────────────────────────────────────────────────────────────────────────────
import time, os, shutil
t0 = time.time()
print('=== Cell 2 started: Copy scripts from Drive ===')

SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
NEEDED = ['multilabel_merged.py', 'multilabel_classifier.py',
          'cnn_classifier.py', 'dataset_chapman.py']

os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

missing = []
for script in NEEDED:
    src = f'{SCRIPTS_DIR}/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{script}')
        print(f'  Copied {script}')
    else:
        missing.append(script)

if missing:
    raise FileNotFoundError(
        f'Missing scripts on Drive: {missing}\n'
        f'Upload them to {SCRIPTS_DIR}/ first.'
    )
print(f'=== Cell 2 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3  ──  CPU runtime  |  Download + extract Chapman ZIP (~2.3 GB)
# Expected: 10-20 min download + 5 min extract
# Safe to re-run: skips if already extracted
# ─────────────────────────────────────────────────────────────────────────────
import time, os, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=== Cell 3 started: Chapman download + extract ===')

CHAPMAN_DIR = '/content/ekg_datasets/chapman'
LOCAL_ZIP   = '/content/chapman.zip'
DRIVE_ZIP   = '/content/drive/MyDrive/EKG_models/chapman.zip'
ZIP_URL     = ('https://physionet.org/static/published-projects/ecg-arrhythmia/'
               'a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0.zip')

os.makedirs(CHAPMAN_DIR, exist_ok=True)
n_existing = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))

if n_existing >= 44000:
    print(f'Chapman already extracted: {n_existing} .mat files - skipping')
else:
    if os.path.exists(DRIVE_ZIP):
        size_gb = os.path.getsize(DRIVE_ZIP) / 1e9
        print(f'[1/3] Copying ZIP from Drive ({size_gb:.1f} GB)...')
        shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'      Done ({time.time()-t0:.0f}s)')
    else:
        print(f'[1/3] Downloading Chapman ZIP from PhysioNet (~2.3 GB)...')
        result = subprocess.run(['wget', '-c', '-q', '--show-progress',
                                 '-O', LOCAL_ZIP, ZIP_URL], check=False)
        if result.returncode != 0 or not os.path.exists(LOCAL_ZIP):
            raise RuntimeError('Download failed. Upload chapman.zip to EKG_models/ manually.')
        size_gb = os.path.getsize(LOCAL_ZIP) / 1e9
        print(f'      Downloaded {size_gb:.1f} GB ({time.time()-t0:.0f}s)')

    print(f'[2/3] Extracting ZIP (45K files, ~5 min)...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', '/content/chapman_extract'], check=True)
    print(f'      Extracted ({time.time()-t0:.0f}s)')

    print(f'[3/3] Moving files to {CHAPMAN_DIR}...')
    extract_base = Path('/content/chapman_extract')
    wfdb_matches = list(extract_base.rglob('WFDBRecords'))
    if wfdb_matches:
        dest = Path(CHAPMAN_DIR) / 'WFDBRecords'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(wfdb_matches[0]), str(dest))
    else:
        for item in extract_base.iterdir():
            shutil.move(str(item), CHAPMAN_DIR)
    shutil.rmtree('/content/chapman_extract', ignore_errors=True)
    os.remove(LOCAL_ZIP)

n_after = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
print(f'Done: {n_after}/45152 .mat files')
if n_after < 44000:
    print(f'WARNING: only {n_after} files — re-run to retry')
print(f'=== Cell 3 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4  ──  CPU runtime  |  Build Chapman index + save to Drive
# Expected: ~5-10 min
# ─────────────────────────────────────────────────────────────────────────────
import time, os, shutil
t0 = time.time()
print('=== Cell 4 started: Build Chapman index ===')
os.chdir('/content')

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
INDEX_PATH   = '/content/ekg_datasets/chapman_index.csv'

if os.path.exists(INDEX_PATH):
    import pandas as pd
    n = len(pd.read_csv(INDEX_PATH))
    print(f'Index already exists: {n} records - skipping build')
else:
    print('Building Chapman index (~5-10 min)...')
    import dataset_chapman
    dataset_chapman.build_chapman_index('/content/ekg_datasets/chapman', INDEX_PATH)
    import pandas as pd
    n = len(pd.read_csv(INDEX_PATH))
    print(f'Index built: {n} records ({time.time()-t0:.0f}s)')

os.makedirs(DRIVE_MODELS, exist_ok=True)
drive_index = f'{DRIVE_MODELS}/chapman_index.csv'
shutil.copy(INDEX_PATH, drive_index)
print(f'Index saved to Drive: {drive_index}')

print(f'=== Cell 4 done ({time.time()-t0:.0f}s) ===')
print('>>> NEXT: Runtime > Change runtime type > T4 GPU > Save, then run Cells 5-8')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5  ──  GPU runtime  |  Verify GPU + mount Drive + copy scripts
# Run this FIRST after switching to GPU runtime
# ─────────────────────────────────────────────────────────────────────────────
import time, subprocess, sys
t0 = time.time()
print('=== Cell 5 started: GPU check + setup ===')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU > Save')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
print('Mounting Drive...')
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)
print('Packages installed.')

import os, shutil
SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Copying scripts...')
for script in ['multilabel_merged.py', 'multilabel_classifier.py',
               'cnn_classifier.py', 'dataset_chapman.py']:
    shutil.copy(f'{SCRIPTS_DIR}/{script}', f'/content/{script}')
    print(f'  Copied {script}')

print(f'=== Cell 5 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6  ──  GPU runtime  |  Restore Chapman from tar + copy PTB-XL to SSD
# Expected: ~5 min (tar) + ~10-15 min (PTB-XL copy)
# ─────────────────────────────────────────────────────────────────────────────
import time, os, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=== Cell 6 started: Restore data to local SSD ===')
os.chdir('/content')

DRIVE_MODELS  = '/content/drive/MyDrive/EKG_models'
CHAPMAN_DIR   = '/content/ekg_datasets/chapman'
CHAPMAN_INDEX = '/content/ekg_datasets/chapman_index.csv'
os.makedirs(CHAPMAN_DIR, exist_ok=True)

# ── Step 1: Restore Chapman files ────────────────────────────────────────────
n_existing = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
if n_existing >= 44000:
    print(f'[1/3] Chapman already present: {n_existing} .mat files - skipping')
else:
    DRIVE_TAR = f'{DRIVE_MODELS}/chapman.tar'
    if not os.path.exists(DRIVE_TAR):
        raise FileNotFoundError(f'chapman.tar not found: {DRIVE_TAR}\nRun CPU Cells 1-4 first.')
    size_gb = os.path.getsize(DRIVE_TAR) / 1e9
    print(f'[1/3] Extracting chapman.tar ({size_gb:.1f} GB) to SSD...')
    subprocess.run(['tar', 'xf', DRIVE_TAR, '-C', '/content'], check=True)
    n_after = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
    print(f'      Chapman restored: {n_after} .mat files ({time.time()-t0:.0f}s)')

# ── Step 2: Restore Chapman index ────────────────────────────────────────────
if os.path.exists(CHAPMAN_INDEX):
    import pandas as pd
    print(f'[2/3] Chapman index present: {len(pd.read_csv(CHAPMAN_INDEX))} records')
else:
    DRIVE_INDEX = f'{DRIVE_MODELS}/chapman_index.csv'
    if os.path.exists(DRIVE_INDEX):
        shutil.copy(DRIVE_INDEX, CHAPMAN_INDEX)
        import pandas as pd
        print(f'[2/3] Chapman index restored: {len(pd.read_csv(CHAPMAN_INDEX))} records ({time.time()-t0:.0f}s)')
    else:
        print('[2/3] Building Chapman index (~5-10 min)...')
        import dataset_chapman
        dataset_chapman.build_chapman_index(CHAPMAN_DIR, CHAPMAN_INDEX)
        import pandas as pd
        print(f'      Index built: {len(pd.read_csv(CHAPMAN_INDEX))} records ({time.time()-t0:.0f}s)')

# ── Step 3: Copy PTB-XL to local SSD ─────────────────────────────────────────
PTBXL_LOCAL = '/content/ekg_datasets/ptbxl'
DRIVE_PTBXL = f'{DRIVE_MODELS}/ptbxl'

if not os.path.exists(DRIVE_PTBXL):
    raise FileNotFoundError(f'PTB-XL not found on Drive: {DRIVE_PTBXL}')

if os.path.exists(PTBXL_LOCAL):
    n_dat = len(list(Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'[3/3] PTB-XL already on SSD: {n_dat} .dat files - skipping')
else:
    n_on_drive = len(list(Path(DRIVE_PTBXL).rglob('*.dat')))
    print(f'[3/3] Copying PTB-XL to SSD ({n_on_drive} .dat files, ~10-15 min)...')
    shutil.copytree(DRIVE_PTBXL, PTBXL_LOCAL)
    n_dat = len(list(Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'      PTB-XL copied: {n_dat} .dat files ({time.time()-t0:.0f}s)')

print(f'=== Cell 6 done ({time.time()-t0:.0f}s) — data ready ===')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7  ──  GPU runtime  |  Train merged 14-class model
# Expected: ~1-2 hrs on T4  |  Streams output line-by-line in real time
# ─────────────────────────────────────────────────────────────────────────────
import time, os, sys, subprocess
t0 = time.time()
print('=== Cell 7 started: Training ===')
os.chdir('/content')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Run Cell 5 first.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Starting multilabel_merged.py — output streams below:', flush=True)
print('-' * 60, flush=True)

# Stream output line-by-line so it appears in real time
proc = subprocess.Popen(
    [sys.executable, '-u', 'multilabel_merged.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('-' * 60, flush=True)
if proc.returncode != 0:
    print(f'Training exited with code {proc.returncode}')
else:
    model_path = '/content/models/ecg_multilabel_v2.pt'
    if os.path.exists(model_path):
        size_mb = os.path.getsize(model_path) / 1e6
        print(f'Model saved: {model_path} ({size_mb:.1f} MB)')
    else:
        print('WARNING: Model file not found')
print(f'=== Cell 7 done ({time.time()-t0:.0f}s) ===')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8  ──  GPU runtime  |  Save trained model to Drive
# ─────────────────────────────────────────────────────────────────────────────
import time, os, shutil
t0 = time.time()
print('=== Cell 8 started: Save model to Drive ===')

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

src = '/content/models/ecg_multilabel_v2.pt'
dst = f'{DRIVE_MODELS}/ecg_multilabel_v2.pt'

if os.path.exists(src):
    print(f'Copying {src} to Drive...')
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'Saved: {dst} ({size_mb:.1f} MB)')
else:
    print(f'WARNING: {src} not found - training may have failed')

print(f'=== Cell 8 done ({time.time()-t0:.0f}s) ===')
print('Download ecg_multilabel_v2.pt from Drive and place in models/ locally.')